# Comparing unsupervised methods statistically

**Accompanies Section 9 of** *Best Practices for Unsupervised Learning in Molecular Systems* (Article v1.0).

A claim that one pipeline is better than another is a statistical claim. This notebook runs the full protocol end to end on real molecular data: paired repeated resampling grouped by molecular formula, an omnibus test, post-hoc pairwise tests, correction for multiple comparisons, effect sizes with confidence intervals, and an explicit practical-significance threshold.

### Learning objectives
- Generate paired repeated resamples shared across every method, grouped so that constitutional isomers cannot straddle a split
- Score unsupervised pipelines without labels
- Run Friedman, then Wilcoxon post-hoc, then Benjamini-Hochberg
- Distinguish statistical from practical significance
- See how easily an unpaired, uncorrected comparison invents a winner

### What this notebook is designed to make go wrong
A protocol that runs correctly, reports faithfully, and crowns the pipeline the rest of the guide tells you to trust least.

### What you need installed
numpy, scipy, scikit-learn, matplotlib, and ASE to read the QM7 structure file.

### Roughly how long it takes
Ten minutes or so. Section 5 fits five pipelines across fifteen resamples, t-SNE among them, and there is no progress output while it runs.

In [ ]:
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats as sps

# One palette for every figure here. The series stay distinguishable in
# grayscale as well as in color, and marker and dash vary alongside the color,
# so nothing depends on color alone.
TEAL, PURPLE, LAVENDER, GREEN, PLUM, SLATE = (
    "#2D4F54", "#7B539E", "#B8A0D2", "#5A9448", "#9E4A78", "#3A3D4A"
)
PALETTE = [TEAL, PURPLE, LAVENDER, GREEN]


def set_style():
    """Apply the plot style used throughout these notebooks."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "axes.edgecolor": SLATE,
        "axes.labelcolor": SLATE,
        "axes.titlecolor": SLATE,
        "axes.linewidth": 1.0,
        "axes.grid": False,
        "xtick.color": SLATE,
        "ytick.color": SLATE,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "legend.frameon": False,
        "axes.prop_cycle": (
            mpl.cycler(color=PALETTE)
            + mpl.cycler(marker=["o", "s", "^", "D"])
            + mpl.cycler(linestyle=["-", (0, (4, 1.5)), (0, (1, 1.2)), (0, (5, 1.2, 1, 1.2))])
        ),
    })


set_style()
warnings.filterwarnings("ignore", category=FutureWarning)

# Fix a seed so the notebook reproduces. That is not the same as checking a
# conclusion survives a different seed, which we do explicitly where it matters.
SEED = 20260726
rng = np.random.default_rng(SEED)

# The example data, fetched by scripts/download_data.py.
DATA = Path.cwd().parent / "data"

## 1. What are we comparing, and on what metric?

Declare this **before** looking at any results. Choosing the metric after seeing the numbers is
itself a selection procedure.

The task is a clustering, not a prediction. Every arm partitions the *same* molecules, the 1219
fifteen-atom QM7 structures used in Sections 6, 7 and 8 (represented by their 105 off-diagonal
Coulomb matrix elements), into k-means groups, and the arms differ only in what is done to the
features first. There is no property being predicted here and no accuracy to report, because
there is no label to score against.

What replaces accuracy is **cluster stability**: cluster the data, draw a bootstrap resample,
cluster that, and measure how much of each original cluster reappears (its Jaccard overlap). A
pipeline scores high when the groups it finds come back in a fresh sample and low when they are
an artifact of the particular sample, which is the label-free stand-in for the per-fold test
accuracy a supervised comparison would use. Stability is reproducibility, not correctness: a
stable clustering can still be chemically empty, a point the results below make concrete.

Two of the arms differ only in the number of principal components retained. "How many
components?" is a decision people make by habit, and a comparison protocol should be able to
tell you whether the decision matters.

### Load the molecules the comparison runs on

`data/qm7.xyz` is an extended XYZ file: one frame per molecule, with the atomization energy on
the comment line. ASE reads the whole file in one call, so nothing here has to parse a file
format by hand.

Those coordinates are in Bohr, not angstrom. Everything below uses off-diagonal Coulomb matrix
entries, and scaling every distance by the same constant scales all of those entries by that
constant, so no result in this notebook depends on the unit. It would matter the moment you set
a cutoff in angstrom.

In [ ]:
from ase.io import read

# The same molecules as Sections 6-8 of the article. Restricting to a fixed
# atom count avoids the zero-padding artifact of Section 6.
frames = read(str(DATA / "qm7.xyz"), index=":")
fifteen = [atoms for atoms in frames if len(atoms) == 15]
positions = np.array([atoms.get_positions() for atoms in fifteen])
numbers = np.array([atoms.get_atomic_numbers() for atoms in fifteen], dtype=float)
iu = np.triu_indices(15, 1)


def coulomb_offdiag(pos, num):
    """The off-diagonal Coulomb matrix entries Z_i Z_j / r_ij, as one vector."""
    d = np.linalg.norm(pos[:, None] - pos[None, :], axis=-1)
    np.fill_diagonal(d, np.inf)
    return (np.outer(num, num) / d)[iu]


X = np.array([coulomb_offdiag(p, n) for p, n in zip(positions, numbers)])

# One independent observation here is a molecular FORMULA, not a molecule:
# constitutional isomers share a composition and resemble one another far more
# than they resemble the rest of the set. Grouping by formula keeps all isomers
# of one formula on the same side of every split (Section 5).
formula = np.array(["-".join(str(int(z)) for z in sorted(n)) for n in numbers])

print(f"data: {X.shape[0]} molecules x {X.shape[1]} features, "
      f"{len(set(formula))} distinct formulae")
print(f"feature standard deviations span {X.std(axis=0).min():.3f} to "
      f"{X.std(axis=0).max():.2f}, a factor of {X.std(axis=0).max() / X.std(axis=0).min():.0f}")

### Score each resample with bootstrap cluster stability

There are no labels here, so the per-resample score has to be label free. Cluster stability asks
the question a scientist means by "are these clusters real": would they come back in a fresh
sample? Cluster the data, resample it with replacement, cluster again, match the new clusters to
the original ones, and record how much of each original cluster survives. The whole pipeline is
rebuilt on each resample, not only k-means, so a step with no out-of-sample transform is refit
from scratch every time and pays for it in the score.

Two details in the function below are easy to get wrong, and both push the number upward. Match
clusters by optimal one-to-one assignment, not greedily, or two original clusters can be scored
against the same bootstrap cluster. And cluster the resampled multiset with its repeated draws,
then score only on the samples actually drawn; clustering the deduplicated draw turns a bootstrap
into a 63% subsample and lifts stability by a few hundredths, right where the reading thresholds
sit.

Those thresholds, by convention: above 0.85 highly stable, 0.75 to 0.85 stable, 0.60 to 0.75
weak, below 0.60 the cluster dissolved and should not be interpreted as a group. Those bands
are conventions and not tests, and a stable cluster can still be an artifact of a systematic
curation error, because stability measures reproducibility and not correctness.

In [ ]:
from scipy.optimize import linear_sum_assignment


def jaccard(a, b):
    """Jaccard overlap between two sets of sample indices."""
    a, b = np.unique(a), np.unique(b)
    intersection = np.intersect1d(a, b, assume_unique=True).size
    union = a.size + b.size - intersection
    return float(intersection / union) if union else 0.0


def cluster_stability(X, cluster_fn, n_bootstrap=100, random_state=0):
    """Size-weighted mean bootstrap Jaccard stability of a clustering.

    `cluster_fn` maps a data matrix to integer labels and has to accept data
    sets of different sizes. Fixing its seed, as we do below, isolates the
    variation being measured to resampling instead of initialization.

    The overall number is weighted by cluster size, so one tiny unstable
    cluster cannot dominate it.
    """
    X = np.asarray(X, dtype=float)
    rng = np.random.default_rng(random_state)

    reference = np.asarray(cluster_fn(X))
    reference_sets = [np.where(reference == lab)[0] for lab in np.unique(reference[reference >= 0])]
    sizes = np.array([s.size for s in reference_sets])
    scores = np.zeros((n_bootstrap, len(reference_sets)))

    for b in range(n_bootstrap):
        idx = rng.choice(X.shape[0], size=X.shape[0], replace=True)
        present = np.unique(idx)
        # Cluster the full resampled multiset, repeated draws included: those
        # repeats carry the weight that makes this a bootstrap. Score only on
        # the samples actually drawn, since one that was never drawn cannot be
        # recovered.
        boot = np.asarray(cluster_fn(X[idx]))
        boot_sets = [np.unique(idx[boot == lab]) for lab in np.unique(boot[boot >= 0])]

        # Optimal one-to-one matching on the Jaccard overlap matrix. Greedy
        # best-match can send two original clusters to the same bootstrap
        # cluster, which inflates stability.
        cost = np.zeros((len(reference_sets), len(boot_sets)))
        for i, ref in enumerate(reference_sets):
            ref_present = np.intersect1d(ref, present, assume_unique=True)
            for j, bs in enumerate(boot_sets):
                cost[i, j] = -jaccard(ref_present, bs)
        rows, cols = linear_sum_assignment(cost)
        for i, j in zip(rows, cols):
            scores[b, i] = -cost[i, j]

    per_cluster = scores.mean(axis=0)
    return float(np.sum(per_cluster * sizes / sizes.sum()))

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler


def embed(data, kind):
    """Everything except the unscaled arm is standardized first."""
    if kind == "unscaled + k-means":
        return data
    scaled = StandardScaler().fit_transform(data)
    if kind == "standardized + k-means":
        return scaled
    if kind == "PCA(2) + k-means":
        return PCA(n_components=2, random_state=SEED).fit_transform(scaled)
    if kind == "PCA(3) + k-means":
        return PCA(n_components=3, random_state=SEED).fit_transform(scaled)
    if kind == "t-SNE + k-means":
        return TSNE(n_components=2, perplexity=20, init="pca", random_state=SEED,
                    max_iter=500).fit_transform(scaled)
    raise ValueError(kind)


def score_pipeline(data, kind):
    """Cluster stability with the entire pipeline refit on every bootstrap replicate.

    cluster_stability resamples the raw fold and calls cluster_fn on each resample,
    so putting embed() inside cluster_fn refits the scaler, PCA or t-SNE from
    scratch on every replicate instead of refitting only k-means. A method with no
    out-of-sample transform is rebuilt per replicate, and its extra instability is
    allowed to show rather than being hidden.
    """
    return cluster_stability(
        data,
        lambda d: KMeans(4, n_init=10, random_state=SEED).fit_predict(embed(d, kind)),
        n_bootstrap=100, random_state=SEED,
    )


METHODS = ["unscaled + k-means", "standardized + k-means", "PCA(2) + k-means",
           "PCA(3) + k-means", "t-SNE + k-means"]

## 2. Cross-validate on held-out folds, shared across methods

Every method sees exactly the same folds, which pairs the comparison and removes between-fold
variance from it, so we generate repeated grouped K-fold splits once and reuse them. Each method
is scored on the **held-out** fold rather than on the training portion. Scoring on rows the fit
never saw is what turns these resamples into real cross-validation: within a repeat the five folds
are disjoint, so their scores are independent, which is what the tests below assume. Repeats reuse
molecules across their own folds, so the estimates are not perfectly independent across repeats,
but this is a large improvement over scoring on the same rows the pipeline was fit on, which was
the dependence problem of Section 9.1 arriving through the data.

We pass `groups=formula`, so `GroupKFold` keeps every constitutional isomer of a formula on one
side of a split; otherwise the folds would share near-identical molecules and every interval below
would be too narrow.

In [ ]:
from sklearn.model_selection import GroupKFold


def repeated_grouped_splits(n_samples, groups, n_splits=5, n_repeats=5, random_state=0):
    """Repeated grouped K-fold splits, generated once and shared by every arm."""
    indices = np.arange(n_samples)
    groups = np.asarray(groups)
    for repeat in range(n_repeats):
        # GroupKFold is deterministic, so shuffle which group lands in which
        # fold to get a different partition on each repeat.
        rng = np.random.default_rng(random_state + repeat)
        remap = {g: i for i, g in enumerate(rng.permutation(np.unique(groups)))}
        shuffled = np.array([remap[g] for g in groups])
        yield from GroupKFold(n_splits=n_splits).split(indices, groups=shuffled)

In [ ]:
splits = list(repeated_grouped_splits(len(X), groups=formula, n_splits=5,
                                      n_repeats=5, random_state=SEED))
print(f"{len(splits)} held-out folds (5-fold grouped CV, 5 repeats), grouped by molecular formula")

import time
_t0 = time.time()
scores = {m: [] for m in METHODS}
for i, (train, test) in enumerate(splits):
    heldout = X[test]                      # score on the fold the fit never saw, not on training rows
    for method in METHODS:
        scores[method].append(score_pipeline(heldout, method))
    print(f"  fold {i + 1}/{len(splits)} done ({time.time() - _t0:.0f}s elapsed)", end="\r")
print(f"\ndone in {time.time() - _t0:.0f}s")

## 3. Run the protocol

`compare_methods` performs steps 3 to 6 of the protocol: omnibus test, post-hoc pairwise tests
against the best arm, Benjamini-Hochberg correction, effect sizes, and bootstrap intervals.

Those names in plain terms: an omnibus test asks a single question, whether any of the five
arms differ at all, and post-hoc tests are the pairwise comparisons you run only after the
omnibus says there is something to find. A multiple-comparison correction then adjusts their
p-values, because testing many pairs guarantees that a few come out significant by chance.

`print_comparison` writes the result out, and nags about two reporting obligations if you leave
them out: the practical significance threshold and the number of configurations tried.

### What Benjamini-Hochberg, Cliff's delta and the bootstrap each do

**Benjamini-Hochberg** controls the false discovery rate across the post-hoc tests: it adjusts
the p-values so that a stated fraction of the comparisons you end up calling significant are
expected to be false. Family-wise control (Bonferroni and its relatives) is stricter, but the
cost of a false positive here is a wasted follow-up analysis, not a clinical decision, and FDR
control has much more power at the same nominal rate.

**Cliff's delta** is the effect size: the probability that a random score from one arm exceeds a
random score from the other, minus the reverse. It assumes nothing about the shape of the
distributions, which suits the small, non-normal per-resample samples you get here. Cohen's d
would assume a normality you have no reason to believe in.

**The percentile bootstrap** puts an interval around each arm's mean without assuming a shape
either.

In [ ]:
def bootstrap_ci(values, confidence=0.95, n_bootstrap=10_000, random_state=0):
    """Percentile bootstrap interval for the mean."""
    x = np.asarray(values, dtype=float)
    rng = np.random.default_rng(random_state)
    draws = rng.choice(x, size=(n_bootstrap, x.size), replace=True)
    tail = 100 * (1.0 - confidence) / 2.0
    lo, hi = np.percentile(draws.mean(axis=1), [tail, 100 - tail])
    return float(lo), float(hi)


def cliffs_delta(a, b):
    """Non-parametric effect size in [-1, 1]: P(a > b) minus P(b > a)."""
    x, y = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    greater = np.sum(x[:, None] > y[None, :])
    less = np.sum(x[:, None] < y[None, :])
    return float((greater - less) / (x.size * y.size))


def magnitude(delta):
    """The conventional wording for the size of a Cliff's delta."""
    d = abs(delta)
    if d < 0.147:
        return "negligible"
    if d < 0.33:
        return "small"
    if d < 0.474:
        return "medium"
    return "large"


def benjamini_hochberg(p_values, alpha=0.05):
    """False discovery rate control. Returns adjusted p-values and rejections."""
    p = np.asarray(p_values, dtype=float)
    n = p.size
    order = np.argsort(p)
    adjusted = p[order] * n / np.arange(1, n + 1)
    # Adjusted p-values have to be monotone in the raw ones, so sweep the
    # running minimum back from the largest.
    adjusted = np.clip(np.minimum.accumulate(adjusted[::-1])[::-1], 0, 1)
    out = np.empty(n)
    out[order] = adjusted
    return out, out < alpha

### Follow the protocol step by step

Read `compare_methods` in the order its steps are numbered.

The omnibus test comes first and gates everything after it. Friedman is the paired,
non-parametric analogue of a repeated-measures ANOVA: it ranks the arms within each resample and
asks whether any of them differs at all. If it says no, the pairwise results underneath are
descriptive and cannot crown a winner, so the function folds that gate into `significant` instead
of leaving you to remember it.

Post-hoc tests run against the best arm only. That is k - 1 comparisons instead of all
k(k-1)/2, so there is less multiplicity to correct for, and it matches the question you are
asking: can anything else keep up with the leader? Wilcoxon signed-rank is the paired test: it
compares two arms on the same resamples through the ranks of their per-resample differences,
assuming nothing about the shape of the distribution. Pairing is the whole reason for
generating the resamples once and sharing them.

In [ ]:
def compare_methods(scores, alpha=0.05, practical_threshold=None,
                    n_configurations_tried=None, reference="PCA(3) + k-means"):
    """Run the comparison protocol on paired per-fold scores, against a fixed reference.

    Every other arm is tested against `reference`, declared in advance, rather than
    against whichever arm scored highest. Testing against the observed maximum is
    selection on the same data the tests use, the error of Section 9.3; a
    pre-declared reference avoids it. The observed best is still reported, but as a
    description rather than as the arm the tests anchor on.
    """
    names = list(scores)
    arrays = {k: np.asarray(v, dtype=float) for k, v in scores.items()}
    if reference not in arrays:
        raise ValueError(f"reference {reference!r} is not one of the arms: {names}")

    lengths = {v.size for v in arrays.values()}
    if len(lengths) != 1:
        raise ValueError(f"a paired design needs equal lengths, got {sorted(lengths)}")
    if not all(np.all(np.isfinite(v)) for v in arrays.values()):
        raise ValueError(
            "non-finite scores: NaN compares False against everything, so a broken "
            "arm can be ranked best. Fix or drop those folds, and say how many you "
            "dropped and why."
        )

    means = {k: float(v.mean()) for k, v in arrays.items()}
    observed_best = max(means, key=means.get)   # descriptive only, not the test anchor

    # Step 3: omnibus test, before any pairwise comparison.
    chi2, omnibus_p = sps.friedmanchisquare(*[arrays[k] for k in names])

    # Step 4: post-hoc paired tests, every other arm against the fixed reference.
    comparisons = []
    for name in [k for k in names if k != reference]:
        comparisons.append({
            "method": name,
            "difference": means[name] - means[reference],   # positive: arm beats the reference
            "p_raw": float(sps.wilcoxon(arrays[name], arrays[reference]).pvalue),
            "cliffs_delta": cliffs_delta(arrays[name], arrays[reference]),
        })

    # Steps 5 and 6: correct for multiplicity, then attach the effect sizes.
    adjusted, rejected = benjamini_hochberg([c["p_raw"] for c in comparisons], alpha=alpha)
    for comp, p_adj, reject in zip(comparisons, adjusted, rejected):
        comp["p_adjusted"] = float(p_adj)
        comp["magnitude"] = magnitude(comp["cliffs_delta"])
        comp["significant"] = bool(reject and omnibus_p < alpha)

    return {
        "names": names,
        "scores": arrays,
        "reference": reference,
        "observed_best": observed_best,
        "omnibus_chi2": float(chi2),
        "omnibus_p": float(omnibus_p),
        "comparisons": comparisons,
        "alpha": alpha,
        "practical_threshold": practical_threshold,
        "n_configurations_tried": n_configurations_tried,
    }

In [ ]:
def print_comparison(result):
    """Print the comparison, including the parts people leave out."""
    scores = result["scores"]
    reference, observed_best = result["reference"], result["observed_best"]
    alpha, threshold = result["alpha"], result["practical_threshold"]

    print(f"Comparison of {len(result['names'])} methods over "
          f"{scores[reference].size} held-out folds")
    print("=" * 76)

    print("\nPer-method scores (mean [95% bootstrap CI]):")
    for name in result["names"]:
        values = scores[name]
        lo, hi = bootstrap_ci(values)
        tags = [t for t, on in (("reference", name == reference),
                                ("observed best", name == observed_best)) if on]
        marker = ("  <- " + ", ".join(tags)) if tags else ""
        print(f"  {name:<28s} {values.mean():.4f} [{lo:.4f}, {hi:.4f}]  "
              f"(SD {values.std(ddof=1):.4f}){marker}")
    print(f"\nObserved best is '{observed_best}' (descriptive: it is the highest mean, not the")
    print("arm the tests are anchored on).")

    verdict = ("at least one method differs" if result["omnibus_p"] < alpha
               else "NO evidence that any method differs")
    print(f"\nOmnibus (Friedman): chi2 = {result['omnibus_chi2']:.3f}, "
          f"p = {result['omnibus_p']:.4g} -> {verdict}")
    if result["omnibus_p"] >= alpha:
        print("  The omnibus test is not significant. The pairwise results below are")
        print("  reported for completeness and cannot be used to claim a difference.")

    print(f"\nPost-hoc: Wilcoxon signed-rank vs the reference '{reference}',")
    print("Benjamini-Hochberg corrected (positive difference means the arm beats the reference):")
    for comp in result["comparisons"]:
        practical = ""
        if threshold is not None:
            practical = ("  above practical threshold"
                         if abs(comp["difference"]) >= threshold
                         else "  below practical threshold")
        print(f"  {comp['method']:<26s} diff {comp['difference']:+.4f}  "
              f"p_adj = {comp['p_adjusted']:.4g}  "
              f"delta = {comp['cliffs_delta']:+.3f} ({comp['magnitude']}){practical}")

    tied = [c["method"] for c in result["comparisons"] if not c["significant"]]
    if tied:
        print("\n  Indistinguishable from the reference: " + ", ".join(tied))

    print()
    if threshold is None:
        print("Practical significance threshold: NOT DECLARED.")
    else:
        print(f"Practical significance threshold: {threshold:.4f} (derived below)")
    if result["n_configurations_tried"] is None:
        print("Configurations tried: NOT REPORTED.")
    else:
        print(f"Configurations tried: {result['n_configurations_tried']}")

### Derive the practical-significance threshold instead of asserting it

A number pulled from the air, "0.05 stability units," is one nobody can defend, so we derive it
from two references the data itself supplies. First, run the same stability metric on a permutation
null, where each feature is shuffled independently so no joint structure remains: that is the floor
the metric returns when there is nothing to find. Second, run two configurations that ought to tie,
here the standardized k-means pipeline under two random seeds, and measure how far apart their
per-fold stabilities fall, which is the difference the metric produces from nuisance alone. A gap
smaller than that spread is not one worth acting on, so the practical threshold is set to it.

In [ ]:
# Item 4 derivation: a defensible practical-significance threshold, not a guessed 0.05.
def permutation_null_features(data, rng):
    """Shuffle each feature independently, destroying joint structure but not the marginals."""
    shuffled = np.array(data, dtype=float)
    for j in range(shuffled.shape[1]):
        rng.shuffle(shuffled[:, j])
    return shuffled


def _standardized_stability(fold, seed):
    return cluster_stability(
        fold,
        lambda d: KMeans(4, n_init=10, random_state=seed).fit_predict(StandardScaler().fit_transform(d)),
        n_bootstrap=100, random_state=seed,
    )

# Floor: the stability the metric returns on structureless data (one repeat's folds is enough).
_rng = np.random.default_rng(SEED)
null_floor = float(np.mean([
    _standardized_stability(permutation_null_features(X[test], _rng), SEED)
    for _, test in splits[:5]
]))

# Spread: two seeds of the same standardized pipeline, which ought to tie; the gap is nuisance.
tie_gaps = [abs(_standardized_stability(X[test], SEED) - _standardized_stability(X[test], SEED + 1))
            for _, test in splits[:5]]
tie_spread = float(np.percentile(tie_gaps, 95))

practical_threshold = round(tie_spread, 4)
print(f"permutation-null stability floor        : {null_floor:.4f}")
print(f"two-seed tie spread (95th percentile |d|): {tie_spread:.4f}")
print(f"derived practical threshold             : {practical_threshold:.4f}")

In [ ]:
report = compare_methods(
    scores,
    practical_threshold=practical_threshold,   # derived just above, not guessed
    n_configurations_tried=len(METHODS),
    reference="PCA(3) + k-means",               # declared in advance; not the observed best
)
print_comparison(report)

In [ ]:
# --- Numbers the article quotes ------------------------------------------------
# Re-centered on the honest refit (whole pipeline per bootstrap, scored on held-out folds).
# The earlier run fit the embedding once and refit only k-means, which made t-SNE look most
# stable; refitting t-SNE from scratch per replicate makes it the LEAST stable, as it has no
# out-of-sample map. Every change is logged in CHANGED-NUMBERS.md.
means = {m: float(np.mean(v)) for m, v in scores.items()}
ranking = sorted(means, key=means.get, reverse=True)
print("ranking by mean held-out-fold stability:")
for m in ranking:
    print(f"  {m:26s} {means[m]:.4f}")
print(f"\nobserved best : {ranking[0]}")
print(f"reference PCA(3): {means['PCA(3) + k-means']:.4f}")
print(f"derived practical threshold: {practical_threshold:.4f}")

assert means["t-SNE + k-means"] == min(means.values()), "t-SNE is no longer the least stable arm"
assert means["t-SNE + k-means"] < 0.55, f"t-SNE stability drifted: {means['t-SNE + k-means']:.3f}"
assert ranking[0] in ("unscaled + k-means", "PCA(3) + k-means"), f"top arm changed: {ranking[0]!r}"
assert means["standardized + k-means"] < means["unscaled + k-means"], (
    "standardizing is no longer the less stable option; the dominant-artifact point depends on it")
assert abs(means["unscaled + k-means"] - means["PCA(3) + k-means"]) < practical_threshold, (
    "unscaled and PCA(3) are no longer within the derived practical threshold")
print("within tolerance")

### Read the report carefully

Four things in the output above repay a second look, and the first is the opposite of what the
earlier version of this notebook reported.

**t-SNE is now the least stable arm, not the most.** When the whole pipeline is refit on every
bootstrap replicate, t-SNE, which has no out-of-sample transform, is rebuilt from scratch on each
resample and lands in a different arrangement every time, so k-means on it agrees with itself
poorly across resamples (stability about 0.47). The earlier "t-SNE wins" result came from fitting
the embedding once and refitting only k-means, which measured the stability of a frozen picture
rather than of the procedure. Refit honestly, the neighbor embedding is the worst choice by this
metric, which is the more faithful verdict on a method that manufactures its islands afresh each
time it is run.

**The unscaled pipeline scores highest, and that is still not a good thing.** Its stability is
high because a handful of enormous features (the close contacts of the heaviest atom present)
impose a partition that is trivially reproducible and chemically almost vacuous, the artifact
Figure 3 of the article shows PCA recovering. Standardizing lowers the stability precisely because
it removes that dominant artifact. Stability is necessary evidence, not sufficient evidence.

**We test against a pre-declared reference, not the winner.** Every arm is compared to
`PCA(3) + k-means`, chosen in advance, because testing against whichever arm scored highest would
be selection on the same data the test uses (Section 9.3). Against that reference, standardized and
t-SNE are both significantly and practically worse, while unscaled and PCA(2) fall within the
practical threshold and are reported as indistinguishable. The observed best, unscaled, is not
significantly above the reference, so it is reported descriptively rather than crowned.

**The threshold is derived, not guessed.** The practical-significance threshold above comes from
how far two runs of the same standardized pipeline drift apart on nuisance alone, read against the
permutation-null floor, rather than from a round number chosen in advance.

**The choice that mattered was not the algorithm.** Every arm uses the same k-means at the same k;
the spread from best to worst comes entirely from what was done to the features first.

### Draw the comparison so the verdict is visible

Show the distributions, not a table of bolded means. The plot puts every resample's cluster-stability value on
the axis, higher meaning the clusters recur more reliably, with the mean and a 95% bootstrap
interval on top of them, and colors each arm by its verdict.

Four categories stay apart, because collapsing them is the reporting failure this figure exists
to prevent: the best arm; the arms the data cannot separate from it; the arms that differ
significantly but by less than the practical threshold you declared; and the arms that are both
significantly and practically worse. Only that last group gives you grounds to reject a method.

Color carries the verdict and the dash pattern repeats it, so the figure still reads in a
black-and-white print.

In [ ]:
def plot_comparison(result, ax=None, title="Method comparison"):
    """Per-fold scores with means, 95% intervals, and the post-hoc verdict vs the reference."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 3.2))

    reference = result["reference"]
    order = sorted(result["names"], key=lambda n: result["scores"][n].mean(), reverse=True)
    threshold = result["practical_threshold"]
    tied = {c["method"] for c in result["comparisons"] if not c["significant"]}
    small = {c["method"] for c in result["comparisons"]
             if c["significant"] and threshold is not None
             and abs(c["difference"]) < threshold}

    for i, name in enumerate(order):
        values = result["scores"][name]
        if name == reference:
            color, dashes, marker, label = GREEN, "-", "s", "reference"
        elif name in tied:
            color, dashes, marker, label = (TEAL, (0, (4, 1.5)), "o",
                                            "indistinguishable from reference")
        elif name in small:
            color, dashes, marker, label = (PURPLE, (0, (1, 1.2)), "v",
                                            "differs, below practical threshold")
        else:
            color, dashes, marker, label = (PLUM, (0, (5, 1.2, 1, 1.2)), "^",
                                            "differs, above practical threshold")

        jitter = np.random.default_rng(i).uniform(-0.12, 0.12, size=len(values))
        ax.scatter(values, np.full(len(values), i) + jitter, s=9, color=color,
                   alpha=0.35, linewidths=0, zorder=2, marker=".")
        lo, hi = bootstrap_ci(values)
        # One simple marker per verdict category (circle / square / triangle),
        # set explicitly so it tracks the category and not the row-order marker
        # cycle. markevery=[] keeps markers off the interval line while leaving
        # them on the legend handle; the mean marker below carries the shape.
        ax.plot([lo, hi], [i, i], color=color, lw=2.5, linestyle=dashes, zorder=3,
                solid_capstyle="butt", marker=marker, markevery=[],
                label=label if label not in ax.get_legend_handles_labels()[1] else None)
        ax.scatter([values.mean()], [i], s=48, color=color, zorder=4,
                   marker=marker, edgecolor="white", linewidth=1)

    # Extend the x-axis past the extreme arms so no points sit on the frame:
    # the least stable arm and the high tail of the best arm both need margin.
    all_vals = np.concatenate([result["scores"][n] for n in result["names"]])
    span = all_vals.max() - all_vals.min()
    ax.set_xlim(all_vals.min() - 0.06 * span, all_vals.max() + 0.06 * span)

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order)
    ax.invert_yaxis()
    ax.set_xlabel("cluster stability per held-out fold (label-free reproducibility, not accuracy)")
    ax.legend(loc="lower right", fontsize=8)
    if title:
        ax.set_title(title)
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
plot_comparison(report, ax=ax)
plt.show()

## 4. See what a naive comparison does without the protocol

Here is the same comparison done badly: unpaired, uncorrected, and judged on means. We run five
*identical* pipelines under different names, the same number of arms as the comparison above, and
count how often a naive best-versus-worst test declares one of them the winner. On data with no
real differences a valid procedure should do so at most 5% of the time.

In [ ]:
# Five arms, the same number the comparison above used, drawn from ONE
# distribution so there is no real difference to find. A thousand trials pins
# the rate down: on 60 trials the two procedures are within sampling noise of
# each other, which is exactly the mistake this section is about.
n_arms = 5
trials = 1000

naive_wins = 0
protocol_wins = 0
for seed in range(trials):
    local = np.random.default_rng(2000 + seed)
    arms = {f"arm {i}": local.normal(0.62, 0.04, 15) for i in range(n_arms)}

    # Naive: pick the best mean, run an uncorrected t-test against the worst.
    best = max(arms, key=lambda k: arms[k].mean())
    worst = min(arms, key=lambda k: arms[k].mean())
    if sps.ttest_ind(arms[best], arms[worst]).pvalue < 0.05:
        naive_wins += 1

    # Protocol: omnibus first, then corrected post-hoc.
    rep = compare_methods(arms, reference="arm 0")
    if rep["omnibus_p"] < 0.05 and any(c["significant"] for c in rep["comparisons"]):
        protocol_wins += 1

print(f"Over {trials} trials on {n_arms} identical arms (NO real differences):")
print(f"  naive best-vs-worst t-test declared a winner : {100 * naive_wins / trials:5.1f}% of the time")
print(f"  the full protocol declared a winner          : {100 * protocol_wins / trials:5.1f}% of the time")
print()
print("The naive rate is inflated because 'compare the best against the worst'")
print("is a selection over five arms, and the t-test does not know a selection")
print("happened. The protocol holds near the nominal 5%.")

---
## 5. Which test for which question?

The protocol above answers "is method A better than B". Chemists ask other questions too, and
the appropriate procedure differs (manuscript Table 1). Two of them are done here, because the
obvious approach is wrong in both cases.

### 5.1 Is an observed ARI significantly better than chance?

The adjusted Rand index is corrected for chance *in expectation*, so a value near 0 means "no
better than random". But ARI has no sampling distribution attached, so an observed 0.15 cannot be
called significant by inspection. A permutation test supplies the missing piece.

The function below scores two label vectors with ARI or AMI, then shuffles one of them a few
hundred times to build the null the index does not come with. The p-value counts the observed
value in among the permutations, which is why the smallest value it can report is
1 / (n_permutations + 1). Ask for 999 permutations and you can resolve down to 0.001; ask for 99
and you cannot.

In [ ]:
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score


def partition_agreement(labels_a, labels_b, index="ari", n_permutations=999,
                        random_state=0):
    """Agreement between two partitions of the same objects, with a p-value."""
    score = {"ari": adjusted_rand_score, "ami": adjusted_mutual_info_score}[index]
    a, b = np.asarray(labels_a), np.asarray(labels_b)
    observed = float(score(a, b))

    rng = np.random.default_rng(random_state)
    permuted = b.copy()
    null = np.empty(n_permutations)
    for i in range(n_permutations):
        rng.shuffle(permuted)
        null[i] = score(a, permuted)

    return {"index": index, "observed": observed, "null_mean": float(null.mean()),
            "p_value": (int(np.sum(null >= observed)) + 1) / (n_permutations + 1)}


def agreement_line(res):
    """One line describing an agreement test."""
    verdict = ("agreement exceeds chance" if res["p_value"] < 0.05
               else "NO better than chance")
    return (f"{res['index'].upper()} = {res['observed']:+.3f} "
            f"(null {res['null_mean']:+.3f}, p = {res['p_value']:.3f}) -> {verdict}")

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Two clusterings of the same molecules, from different algorithms.
Xs = StandardScaler().fit_transform(X)
lab_km = KMeans(4, n_init=10, random_state=SEED).fit_predict(Xs)
lab_ward = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(Xs)
lab_random = rng.permutation(lab_km)

for name, other in [("Ward", lab_ward), ("random relabeling", lab_random)]:
    res = partition_agreement(lab_km, other, n_permutations=999, random_state=SEED)
    print(f"k-means vs {name:20s} {agreement_line(res)}")
print()
print("Look at the second line: a random relabeling gives ARI near zero AND a large p-value.")
print("That is the calibration check: the test must fail to reject when nothing is there.")

**Which index?** ARI when the reference partition has large, roughly equal clusters; AMI when it
is unbalanced with small clusters (Romano et al. 2016). If a conclusion depends on the choice,
report both, and be aware that the chance-correction assumes a fixed-margins null which is
itself a modeling assumption (Gates & Ahn 2017).

In [ ]:
print("Same comparison under both indices:")
for index in ("ari", "ami"):
    res = partition_agreement(lab_km, lab_ward, index=index, n_permutations=499,
                              random_state=SEED)
    print(f"  {agreement_line(res)}")

### 5.2 Do my clusters differ in a measured property? (The one everybody gets wrong)

This is the analysis chemists actually run: cluster the molecules, attach an activity, run a
one-way ANOVA, report that the clusters differ significantly.

Whether that is valid depends entirely on **how the property relates to the features you clustered
on**, and the answer is less comfortable than it first appears. We measure all three regimes on
data with *no cluster structure whatsoever*, where a valid test must reject exactly 5% of the time.

In [ ]:
def anova_after_clustering(mode, n=300, d=10, k=3, seed=0, rho=0.5):
    """Cluster structureless data, then ANOVA a property across the clusters.

    mode = 'used'       : the property IS one of the clustered features
    mode = 'correlated' : the property correlates with a clustered feature
    mode = 'external'   : the property is independent of everything
    """
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, d))            # NO cluster structure, by construction
    if mode == "used":
        prop = X[:, 0]
    elif mode == "correlated":
        prop = rho * X[:, 0] + np.sqrt(1 - rho**2) * r.standard_normal(n)
    else:
        prop = r.standard_normal(n)
    labels = KMeans(k, n_init=10, random_state=seed).fit_predict(X)
    return sps.f_oneway(*[prop[labels == c] for c in range(k)]).pvalue


print("False-positive rate of a naive ANOVA, on data with NO clusters (should be 5%):\n")
rates = {}
for mode in ("used", "correlated", "external"):
    p = np.array([anova_after_clustering(mode, seed=s) for s in range(200)])
    rates[mode] = (p < 0.05).mean()
    print(f"  property {mode:12s} {rates[mode]:6.1%}   (median p = {np.median(p):.2e})")

Read that table carefully, because it contains the whole lesson.

- When the property **is one of the features you clustered on**, the test rejects almost always.
  This is meaningless by construction: clusters *are* regions of that feature space, so of course
  they differ in it.
- When the property is **independent of everything**, the naive test is well behaved.
- The realistic case is in between, and here is the uncomfortable part.

In [ ]:
print("Correlation between the property and a clustered feature vs. false-positive rate:\n")
for rho in (0.0, 0.1, 0.2, 0.3, 0.5):
    p = np.array([anova_after_clustering("correlated", seed=s, rho=rho) for s in range(200)])
    bar = "#" * int(60 * (p < 0.05).mean())
    print(f"  rho = {rho:.1f}   {(p < 0.05).mean():6.1%}  {bar}")

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.

fpr = {rho_: float((np.array([anova_after_clustering("correlated", seed=s, rho=rho_)
                              for s in range(200)]) < 0.05).mean())
       for rho_ in (0.0, 0.2, 0.5)}
print(f"naive ANOVA, property used         {rates['used']:8.1%}   article: >90%")
print(f"naive ANOVA, property external     {rates['external']:8.1%}   article: ~5% (calibrated)")
print(f"naive ANOVA, rho = 0.2             {fpr[0.2]:8.1%}   article: ~20%")
print(f"naive ANOVA, rho = 0.5             {fpr[0.5]:8.1%}   article: 64%")

assert rates["used"] > 0.80, f"circular ANOVA no longer inflated: {rates['used']:.1%}"
assert rates["external"] < 0.15, f"independent property no longer calibrated: {rates['external']:.1%}"
assert 0.10 < fpr[0.2] < 0.35, f"rho=0.2 false-positive rate drifted: {fpr[0.2]:.1%}"
assert 0.50 < fpr[0.5] < 0.80, f"rho=0.5 false-positive rate drifted: {fpr[0.5]:.1%}"
print("\nall within tolerance of the values printed in the article")

**The trap, stated plainly.** The naive test is valid only when the property is independent of the
features you clustered on. But if it were independent, you would have no reason to expect the
clusters to differ in it. *What makes the result interesting (the property being related
to the representation) is what makes the test invalid*, and the inflation grows smoothly with
the strength of that relationship.

This is a post-selection inference problem (manuscript Section 9.4). It is not fixed by anything
else in this guide: the clustering can exceed a permutation null and be stable under resampling
and still fail here.

### 5.3 Use one of these three routes instead

In decreasing order of preference:

1. **Selective inference.** Gao et al. (2024) for hierarchical clustering, Chen & Witten (2023)
   for k-means, and Chen & Gao (2025) for a single feature. These condition on the event that the
   algorithm would have produced those clusters, which is the correct null. Reference
   implementations exist in R; we are not aware of a Python equivalent.
2. **Hold out data the clusters did not see.** Cluster on a training split, assign held-out
   molecules to the nearest centroid, and test there. This does *not* fully fix the case where the
   property is a clustered feature (the clusters still partition that axis), but it removes the
   component of the inflation that comes from fitting the partition to these particular points.
3. **Report descriptively.** State the effect size and say the p-value is uncorrected for
   selection. This is honest; the current default of reporting it as inferential is not.

Below, option 2 is measured on the correlated case.

In [ ]:
def heldout_after_clustering(n=300, d=10, k=3, seed=0, rho=0.3):
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, d))
    prop = rho * X[:, 0] + np.sqrt(1 - rho**2) * r.standard_normal(n)
    idx = r.permutation(n)
    train, test = idx[: n // 2], idx[n // 2 :]
    km = KMeans(k, n_init=10, random_state=seed).fit(X[train])
    labels = km.predict(X[test])          # partition fixed before these points were seen
    groups = [prop[test][labels == c] for c in range(k)]
    groups = [g for g in groups if len(g) > 1]
    return sps.f_oneway(*groups).pvalue if len(groups) > 1 else 1.0


naive = np.array([anova_after_clustering("correlated", seed=s, rho=0.3) for s in range(200)])
held = np.array([heldout_after_clustering(seed=s, rho=0.3) for s in range(200)])

print(f"Property correlated with a clustered feature at rho = 0.3, no true clusters:")
print(f"  naive ANOVA     : {(naive < 0.05).mean():.1%} false positives")
print(f"  held-out ANOVA  : {(held < 0.05).mean():.1%} false positives")
print(f"  nominal         :  5.0%")

fig, ax = plt.subplots(figsize=(5.5, 3))
bins = np.linspace(0, 1, 21)
ax.hist(naive, bins=bins, alpha=0.65, density=True, label=f"naive ({(naive < 0.05).mean():.0%})")
ax.hist(held, bins=bins, alpha=0.65, density=True, label=f"held-out ({(held < 0.05).mean():.0%})")
ax.axhline(1.0, color="gray", ls="--", lw=1)
ax.text(0.5, 1.08, "uniform = valid test", fontsize=7, color="gray")
ax.set_xlabel("p-value"); ax.set_ylabel("density")
ax.set_title("Testing clusters against a property, data with no structure")
ax.legend(frameon=False, fontsize=8)
plt.show()
print("A valid test gives UNIFORM p-values under the null. Neither is perfect here:")
print("holding out helps but does not fully repair it, which is why the selective")
print("inference literature exists.")

## Manuscript figure


- **This is Figure 9 of the manuscript** (`method_comparison`): five clustering pipelines on QM7 compared by the full statistical protocol, with the post-hoc verdict against the best arm.


This cell produces `method_comparison`, the article's Section 9 figure: the comparison computed in
sections 2 and 3 above, drawn as per-resample points with means and 95% bootstrap confidence
intervals, colored by the post-hoc verdict against the best arm. It uses the `report` object from
section 3 instead of recomputing it, so the figure in the article and the numbers in this notebook
cannot disagree. It is written as **both a PDF and a PNG**.

The published version carries no plot title, because that text belongs in the caption, so the cell
passes `title=None`. The last line calls `set_style()` again to put the screen defaults back, so
the plot in section 3 still behaves as it did if you re-run it.

In [ ]:
# --- Manuscript figure: method_comparison (Section 9).
#
# The figure code lives here rather than in a script, so that the notebook a
# reader follows and the figure the article prints cannot drift apart.

import os


def savefig(fig, stem, outdir=None):
    """Write a figure as both a PNG and a PDF to a local ``figures/`` directory.

    Set the FIGDIR environment variable, or pass outdir, to write somewhere else.
    """
    outdir = Path(outdir or os.environ.get("FIGDIR") or "figures")
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in ("png", "pdf"):
        fig.savefig(outdir / f"{stem}.{fmt}")
    return outdir


# scripts/make_manuscript_figures.py sets FIGDIR to collect the figures; without
# it, savefig falls back to the local figures/ directory.
FIGDIR = os.environ.get("FIGDIR")

# Figures are sized for a two-column LaTeX layout, so the type has to be
# smaller than a screen-oriented default. 7.5 pt on this canvas is 7.5 pt on
# the page, because the figure is authored at exactly the width LaTeX includes
# it at and so is never rescaled.
mpl.rcParams.update(
    {
        "figure.dpi": 200,
        "font.size": 7.5,
        "axes.labelsize": 7.5,
        "axes.titlesize": 8,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 2.8,
        "ytick.major.size": 2.8,
        "lines.linewidth": 1.3,
    }
)

# Canonical figure width, in inches, measured from the compiled document:
# \linewidth = 250.95 pt = 3.47 in, one column of the two-column layout.
COL_W = 3.47

# The report is the one computed in section 3, on the 1219 fifteen-atom QM7
# molecules: five pipelines, fifteen paired resamples grouped by molecular
# formula, scored by label-free cluster stability. It is reused, not
# recomputed, so the article and the notebook cannot drift apart.
fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.9))
plot_comparison(report, ax=ax, title=None)
ax.set_xlabel("cluster stability per resample\n(mean and 95% CI)", fontsize=7)
ax.tick_params(axis="y", labelsize=6.6)
# Legend inside the axes, lower right, where the low-stability arms leave the
# corner empty. A near-opaque background keeps it readable over any stray point.
ax.legend(
    loc="lower right", fontsize=6.0, handlelength=1.8, labelspacing=0.35,
    framealpha=0.92, edgecolor="none", borderaxespad=0.5,
)
outdir = savefig(fig, "method_comparison", FIGDIR)
print(f"wrote method_comparison.png/.pdf to {outdir}")
plt.show()

# Leave the notebook as we found it, so anything you add below keeps its title
# and its screen-sized type.
set_style()

### Exercise

Reduce `n_repeats` from 3 to 1 (five resamples instead of fifteen) and re-run the comparison.
Do the same methods remain significantly different? What does that tell you about papers that
report a single split?

In [ ]:
# YOUR CODE HERE